In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from fontTools.ttLib.woff2 import bboxFormat

import utils

plt.style.use('default')

In [ ]:
CI = 95  # 95 % CI
ALPHA = 1 - CI / 100.0

df = utils.union_read_csv('data/Gson_iterations.csv', 'data/Lang_iterations.csv', 'data/JacksonCore_iterations.csv')
df = df[df['RUN_COMPLETED'] == True]
# Do not count iteration 0 twice per run
df['TYPE'] = df.apply(lambda row: 'INITIAL' if row['ITERATION'] == 0 else row['FEEDBACK_TYPE'], axis=1)
df = df[['RUN_UUID', 'CLASS_NAME', 'TYPE', 'ITERATION', 'REAL_BUG_RESULT']].drop_duplicates(
    ['RUN_UUID', 'CLASS_NAME', 'TYPE', 'ITERATION'])

In [ ]:
per_iteration = (df.groupby(['ITERATION', 'REAL_BUG_RESULT'], dropna=False)
                 .aggregate(count=('TYPE', 'count'))
                 .unstack(fill_value=0)
                 .rename(columns={np.nan: 'N/A'}))
per_iteration

In [ ]:
figsize = (utils.A5_LANDSCAPE_INCHES[0], utils.A5_LANDSCAPE_INCHES[1] / 3 * 2)

fig, axes = plt.subplots(1, 1, figsize=figsize, dpi=300)
fig.suptitle('Erkennung von realen Bugs')

axes.bar(
    per_iteration.index,
    per_iteration[('count', 'DETECTED')],
    label='Erkannt',
    color='C1',
)
axes.bar(
    per_iteration.index,
    per_iteration[('count', 'UNDETECTED')],
    bottom=per_iteration[('count', 'DETECTED')],
    label='Nicht erkannt',
    color='C0',
)
axes.bar(
    per_iteration.index,
    per_iteration[('count', 'N/A')],
    bottom=per_iteration[('count', 'DETECTED')] + per_iteration[('count', 'UNDETECTED')],
    label='N/A',
    color='C3',
)

tops = per_iteration[('count', 'DETECTED')] + per_iteration[('count', 'UNDETECTED')] + per_iteration[('count', 'N/A')]
for i, bar in enumerate(axes.containers):
    bbox = dict(facecolor=('w', .5), edgecolor='none', pad=.2, boxstyle='round') if i == 2 else None
    axes.bar_label(bar, label_type='center', color='w' if i == 1 else 'k', fmt=lambda x: '' if x == 0 else f'{x:,.0f}'.replace(',', ' '), bbox=bbox, size='small')
    axes.bar_label(bar, label_type='edge', padding=4, fmt=lambda x: '' if i != 2 else f'{x:,.0f}'.replace(',', ' '))

axes.margins(y=.1)
axes.set_xlabel('Iteration')
axes.set_xticks(per_iteration.index)
axes.yaxis.set_major_formatter(ticker.FuncFormatter(lambda val, pos: f'{val:,.0f}'.replace(',', ' ')))
axes.grid(linestyle='--', linewidth=.5, alpha=0.5, axis='y')
axes.legend()

plt.show()
fig.savefig(f'out/bug-detections.svg', bbox_inches='tight')